# Time-Series Forecasting Baselines
**Notebook 1 of 2** — Baseline benchmark on 6 standard datasets.

Models: NLinear, DLinear, LSTM, DeepAR, Transformer, Informer, PatchTST, FEDformer, TimesNet

Datasets: ETTh1, ETTh2, ETTm1, ETTm2, Weather, ECL


In [ ]:
# ── 0. Update DATA_ROOT to match your environment ─────────────────────────────
DATA_ROOT = '/kaggle/input/datasets/rahuldray12324/six-datasets/all_six_datasets'
# DATA_ROOT = '/path/to/all_six_datasets'   # ← change for local runs


In [ ]:
# ── 1. Imports & Config ───────────────────────────────────────────────────────
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.preprocessing import StandardScaler
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

BATCH    = 64
SEQ_LEN  = 96
PRED_LEN = 24
EPOCHS   = 10
PATIENCE = 3
LR       = 1e-3
N_GPUS   = max(1, torch.cuda.device_count())
print(f'GPUs detected: {N_GPUS}')

DATA_PATHS = {
    k: f'{DATA_ROOT}/{k}/Y_df.csv'
    for k in ['ETTh1','ETTh2','ETTm1','ETTm2','Weather','ECL']
}


In [ ]:
# ── 2. Dataset ────────────────────────────────────────────────────────────────
class TSDataset(Dataset):
    def __init__(self, data, seq_len, pred_len):
        self.x  = torch.tensor(data, dtype=torch.float32)
        self.sl = seq_len
        self.pl = pred_len
    def __len__(self):
        return len(self.x) - self.sl - self.pl + 1
    def __getitem__(self, i):
        return self.x[i:i+self.sl], self.x[i+self.sl:i+self.sl+self.pl]

def load_data(name):
    df  = pd.read_csv(DATA_PATHS[name])
    col = 'y' if 'y' in df.columns else next(
        c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]))
    vals = df[col].values.astype(np.float32)
    n    = len(vals)
    t1, t2 = int(.7*n), int(.8*n)
    sc = StandardScaler()
    tr = sc.fit_transform(vals[:t1].reshape(-1,1)).flatten()
    va = sc.transform(vals[t1:t2].reshape(-1,1)).flatten()
    te = sc.transform(vals[t2:].reshape(-1,1)).flatten()
    mad = float(np.mean(np.abs(np.diff(tr))))  # MASE denominator
    def dl(d, sh): return DataLoader(
        TSDataset(d, SEQ_LEN, PRED_LEN), BATCH,
        shuffle=sh, num_workers=2, pin_memory=True)
    return dl(tr,True), dl(va,False), dl(te,False), mad


In [ ]:
# ── 3. Model Definitions ──────────────────────────────────────────────────────

# ─── NLinear / DLinear ───────────────────────────────────────────────────────
class NLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(SEQ_LEN, PRED_LEN)
    def forward(self, x):
        return self.linear(x - x[:,-1:])

class DLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.avg   = nn.AvgPool1d(25, stride=1, padding=12)
        self.trend = nn.Linear(SEQ_LEN, PRED_LEN)
        self.seas  = nn.Linear(SEQ_LEN, PRED_LEN)
    def forward(self, x):
        t = self.avg(x.unsqueeze(1)).squeeze(1)[:,:SEQ_LEN]
        return self.trend(t) + self.seas(x - t)

# ─── LSTM / DeepAR ───────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 128, 2, batch_first=True, dropout=0.1)
        self.fc   = nn.Linear(128, PRED_LEN)
    def forward(self, x):
        out, _ = self.lstm(x.unsqueeze(-1))
        return self.fc(out[:,-1])

class DeepARModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm  = nn.LSTM(1, 128, 2, batch_first=True, dropout=0.1)
        self.mu_h  = nn.Linear(128, PRED_LEN)
        self.sig_h = nn.Linear(128, PRED_LEN)
    def forward(self, x):
        out, _ = self.lstm(x.unsqueeze(-1))
        h = out[:,-1]
        return self.mu_h(h), F.softplus(self.sig_h(h)) + 1e-6

# ─── Positional Encoding ─────────────────────────────────────────────────────
class PosEnc(nn.Module):
    def __init__(self, d, max_len=512):
        super().__init__()
        pe  = torch.zeros(max_len, d)
        pos = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0,d,2).float() * (-math.log(10000)/d))
        pe[:,0::2] = torch.sin(pos*div)
        pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:,:x.size(1)]

# ─── Transformer ─────────────────────────────────────────────────────────────
class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Linear(1, 128)
        self.pos = PosEnc(128)
        enc = nn.TransformerEncoderLayer(128, 4, 256, 0.1, batch_first=True, norm_first=True)
        self.tf  = nn.TransformerEncoder(enc, 2)
        self.fc  = nn.Linear(128*SEQ_LEN, PRED_LEN)
    def forward(self, x):
        B = x.size(0)
        return self.fc(self.tf(self.pos(self.emb(x.unsqueeze(-1)))).reshape(B,-1))

# ─── Informer ────────────────────────────────────────────────────────────────
class Informer(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Linear(1, 128)
        self.pos = PosEnc(128)
        enc = nn.TransformerEncoderLayer(128, 4, 256, 0.1, batch_first=True, norm_first=True)
        self.tf  = nn.TransformerEncoder(enc, 2)
        self.fc  = nn.Linear(128*SEQ_LEN, PRED_LEN)
    def forward(self, x):
        B = x.size(0)
        return self.fc(self.tf(self.pos(self.emb(x.unsqueeze(-1)))).reshape(B,-1))

# ─── PatchTST ────────────────────────────────────────────────────────────────
class PatchTST(nn.Module):
    def __init__(self, patch_len=16, stride=8):
        super().__init__()
        self.pl = patch_len; self.st = stride
        n = (SEQ_LEN - patch_len) // stride + 1
        self.emb = nn.Linear(patch_len, 128)
        self.cls = nn.Parameter(torch.zeros(1,1,128))
        self.pos = nn.Parameter(torch.randn(1,n+1,128)*0.02)
        enc = nn.TransformerEncoderLayer(128, 4, 256, 0.1, batch_first=True, norm_first=True)
        self.tf  = nn.TransformerEncoder(enc, 2)
        self.fc  = nn.Linear(128, PRED_LEN)
    def forward(self, x):
        B = x.size(0)
        tok = self.emb(x.unfold(-1, self.pl, self.st))
        tok = torch.cat([self.cls.expand(B,-1,-1), tok], 1) + self.pos
        return self.fc(self.tf(tok)[:,0])

# ─── FEDformer ───────────────────────────────────────────────────────────────
class FEDformer(nn.Module):
    def __init__(self, d=128, n_modes=64, n_layers=2):
        super().__init__()
        self.emb   = nn.Linear(1, d)
        self.W_r   = nn.ParameterList(
            [nn.Parameter(torch.randn(n_modes,d,d)*0.02) for _ in range(n_layers)])
        self.W_i   = nn.ParameterList(
            [nn.Parameter(torch.randn(n_modes,d,d)*0.02) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])
        self.ffs   = nn.ModuleList([
            nn.Sequential(nn.Linear(d,256),nn.GELU(),nn.Linear(256,d))
            for _ in range(n_layers)])
        self.proj  = nn.Linear(d*SEQ_LEN, PRED_LEN)
    def forward(self, x):
        B, T = x.shape
        h = self.emb(x.unsqueeze(-1))
        for Wr, Wi, norm, ff in zip(self.W_r, self.W_i, self.norms, self.ffs):
            xf  = torch.fft.rfft(h, dim=1)
            m   = min(Wr.size(0), xf.size(1))
            out = xf.clone()
            out[:,:m] = (
                torch.einsum('bmd,mdo->bmo', xf[:,:m].real, Wr[:m])
                + 1j*torch.einsum('bmd,mdo->bmo', xf[:,:m].imag, Wi[:m]))
            h2 = torch.fft.irfft(out, n=T, dim=1)
            h  = norm(h + h2)
            h  = norm(h + ff(h))
        return self.proj(h.reshape(B,-1))

# ─── TimesNet ────────────────────────────────────────────────────────────────
class TimesBlock(nn.Module):
    def __init__(self, d=64, d_ff=128, top_k=5):
        super().__init__()
        self.top_k = top_k
        self.conv  = nn.Sequential(
            nn.Conv2d(d, d_ff, 3, padding=1), nn.GELU(),
            nn.Conv2d(d_ff, d, 3, padding=1))
    def forward(self, x):
        B, T, D = x.shape
        amps = torch.fft.rfft(x.mean(-1), dim=-1).abs()[:,1:]
        top_p = amps.topk(min(self.top_k, amps.size(-1)), dim=-1).indices + 1
        outs  = []
        for p in range(top_p.size(-1)):
            period = max(1, int(top_p[:,p].float().mean().round().item()))
            pl_len = math.ceil(T / period)
            pad    = period * pl_len - T
            xp = F.pad(x, (0,0,0,pad)).reshape(B, pl_len, period, D).permute(0,3,1,2)
            xp = self.conv(xp).permute(0,2,3,1).reshape(B,-1,D)[:,:T]
            outs.append(xp)
        return torch.stack(outs,-1).mean(-1)

class TimesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb    = nn.Linear(1, 64)
        self.blocks = nn.ModuleList([TimesBlock() for _ in range(2)])
        self.norms  = nn.ModuleList([nn.LayerNorm(64) for _ in range(2)])
        self.proj   = nn.Linear(64*SEQ_LEN, PRED_LEN)
    def forward(self, x):
        B = x.size(0)
        h = self.emb(x.unsqueeze(-1))
        for blk, norm in zip(self.blocks, self.norms):
            h = norm(h + blk(h))
        return self.proj(h.reshape(B,-1))

print('All model classes defined ✓')


In [ ]:
# ── 4. Losses & Metrics ───────────────────────────────────────────────────────
def gaussian_nll(mu, var, y):
    return (0.5*torch.log(var) + 0.5*(y-mu).pow(2)/var).mean()

def crps_gaussian(mu, sig, y):
    n = torch.distributions.Normal(0., 1.)
    z = (y-mu) / sig.clamp(min=1e-8)
    return (sig*(z*(2*n.cdf(z)-1) + 2*n.log_prob(z).exp()
               - 1/math.sqrt(math.pi))).mean().item()

def compute_metrics(pred, var, tgt, mad):
    mae   = (pred-tgt).abs().mean().item()
    rmse  = ((pred-tgt)**2).mean().sqrt().item()
    smape = (2*(pred-tgt).abs()/(pred.abs()+tgt.abs()+1e-8)).mean().item()*100
    mase  = mae / (mad+1e-8)
    std   = var.clamp(1e-10).sqrt()
    crps  = crps_gaussian(pred, std, tgt)
    nll   = gaussian_nll(pred, var.clamp(1e-10), tgt).item()
    return mae, rmse, smape, mase, crps, nll


In [ ]:
# ── 5. Training Engine ────────────────────────────────────────────────────────
def train_model(name, gpu_id, model_cls, prob=False):
    device = torch.device(f'cuda:{gpu_id}' if torch.cuda.is_available() else 'cpu')
    t0     = time.time()
    tr_dl, va_dl, te_dl, mad = load_data(name)
    model  = model_cls().to(device)
    n_par  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
    scaler = GradScaler()
    best_v, best_st, pat = float('inf'), None, 0

    for ep in range(1, EPOCHS+1):
        model.train()
        for x, y in tr_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with autocast():
                out  = model(x)
                loss = gaussian_nll(*out, y) if prob else (out-y).abs().mean()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.)
            scaler.step(opt); scaler.update()
        sched.step()
        model.eval(); vl = 0.
        with torch.no_grad():
            for x, y in va_dl:
                x, y = x.to(device), y.to(device)
                out  = model(x)
                vl  += (gaussian_nll(*out, y) if prob else (out-y).abs().mean()).item()
        vl /= max(len(va_dl), 1)
        if vl < best_v - 1e-6:
            best_v=vl; pat=0
            best_st = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pat += 1
            if pat >= PATIENCE: break

    if best_st:
        model.load_state_dict({k: v.to(device) for k,v in best_st.items()})
    model.eval()
    all_mu, all_var, all_y = [], [], []
    with torch.no_grad():
        for x, y in te_dl:
            x   = x.to(device)
            out = model(x)
            if prob: mu, var = out
            else:    mu, var = out, torch.ones_like(out)
            all_mu.append(mu.cpu()); all_var.append(var.cpu()); all_y.append(y)
    mu_t  = torch.cat(all_mu)
    var_t = torch.cat(all_var).clamp(1e-10)
    y_t   = torch.cat(all_y)
    mae, rmse, smape, mase, crps, nll = compute_metrics(mu_t, var_t, y_t, mad)
    elapsed = round(time.time()-t0, 1)
    mname   = model.__class__.__name__
    print(f'  ✓ [{mname}/{name}]  MAE={mae:.4f}  RMSE={rmse:.4f}  '
          f'sMAPE={smape:.2f}%  CRPS={crps:.4f}  NLL={nll:.4f}  '
          f'params={n_par:,}  time={elapsed}s')
    return dict(model=mname, dataset=name,
                mae=round(mae,4), rmse=round(rmse,4), smape=round(smape,4),
                mase=round(mase,4),
                crps=round(crps,4) if prob else float('nan'),
                nll=round(nll,4)   if prob else float('nan'),
                n_params=n_par, train_time_sec=elapsed)


In [ ]:
# ── 6. Run all baselines ──────────────────────────────────────────────────────
DATASETS = ['ETTh1','ETTh2','ETTm1','ETTm2','Weather','ECL']

REGISTRY = [
    ('NLinear',     NLinear,         False),
    ('DLinear',     DLinear,         False),
    ('LSTM',        LSTMModel,       False),
    ('DeepAR',      DeepARModel,     True),
    ('Transformer', TransformerModel,False),
    ('Informer',    Informer,        False),
    ('PatchTST',    PatchTST,        False),
    ('FEDformer',   FEDformer,       False),
    ('TimesNet',    TimesNet,        False),
]

all_results = []

for mname, cls, prob in REGISTRY:
    print(f'\n{"="*55}\n{mname}\n{"="*55}')
    gpu_map = {ds: i % N_GPUS for i,ds in enumerate(DATASETS)}
    with ThreadPoolExecutor(max_workers=N_GPUS) as ex:
        futures = {ex.submit(train_model, ds, gpu_map[ds], cls, prob): ds
                   for ds in DATASETS}
        for fut in as_completed(futures):
            try:
                r = fut.result(); r['model'] = mname; all_results.append(r)
            except Exception as e:
                print(f'  [ERROR] {futures[fut]}: {e}')

df = pd.DataFrame(all_results).sort_values(['model','dataset']).reset_index(drop=True)
df.to_csv('baseline_results.csv', index=False)
print('\n\nSaved → baseline_results.csv')
df


In [ ]:
# ── 7. Summary pivot table ────────────────────────────────────────────────────
pivot = df.pivot_table(index='dataset', columns='model', values='mae').round(4)
print('MAE summary (lower is better):')
pivot
